# 03 · DDP Fundamentals：多个进程怎样保持同一数学语义

**本节问题：** 数据切分与梯度 AllReduce 怎样让各 rank 得到一致参数？

完成后你应该能够：

- 解释 rank/world size/process group
- 检查 DistributedSampler 覆盖
- 区分 strong 与 weak scaling

前置阅读：[模块 README](../03_distributed_training/README.md) · [术语表](../docs/concepts/distributed-systems-glossary.md)


## 运行状态卡

默认 `reference` 可在无 GPU 电脑上 Run All。改为 `local` 或 `gpu` 才会启动 runner。

In [ ]:
# 第一处可编辑配置：reference | local | gpu
MODE = "reference"

import sys
from pathlib import Path

notebook_dir = Path("notebooks") if Path("notebooks/_support").is_dir() else Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from _support.artifacts import load_artifact
from _support.context import create_context
from _support.plots import bar_chart
from _support.runner import run_command

ctx = create_context("03_ddp_fundamentals", MODE)
ctx.card()


## 运行前预测

先写下你的预测。不要担心猜错；后面需要指出证据支持或推翻了哪一部分。


In [ ]:
PREDICTION = "我预计……，因为……"
PREDICTION

## 最小观察

这个单元只暴露关键中间状态；正式算法仍来自项目源码。

In [ ]:
samples = list(range(10))
world_size = 3
shards = {rank: samples[rank::world_size] for rank in range(world_size)}
shards

## 正确性门与参考证据

读取已提交 JSON；字段缺失时立即停止，不把缺失值解释成 0。

In [ ]:
artifact_path = ctx.repo_root / '03_distributed_training/results/module03_summary.json'
artifact = load_artifact(artifact_path, required=['schema_version', 'gates', 'all_executable_gates_passed'])
print("证据来源：仓库参考结果", artifact_path.relative_to(ctx.repo_root))
print("顶层字段：", sorted(artifact))


In [ ]:
gates = artifact['gates']
print('通过 gate：', sum(gates.values()), '/', len(gates))
print('全部可执行 gate 通过：', artifact['all_executable_gates_passed'])

In [ ]:
gates = artifact['gates']
bar_chart(['passed', 'not passed'], [sum(gates.values()), len(gates)-sum(gates.values())], title='正确性与实验 gate', ylabel='gate 数量')

## 本地/正式实验

命令使用参数列表在独立子进程中执行，日志和产物只写入 `_runs/`。

In [ ]:
commands = {
    "local": [sys.executable, '03_distributed_training/benchmarks/run_correctness.py', '--experiment', 'gradient', '--world-size', '2', '--output', str(ctx.output_dir / 'gradient.json')],
    "gpu": [sys.executable, '03_distributed_training/benchmarks/run_scaling.py', '--config', '03_distributed_training/configs/gpu_scaling_smoke.toml', '--output', str(ctx.output_dir / 'scaling.json')],
}
command = commands.get(ctx.mode)
if ctx.mode == "reference":
    print("reference 模式：只读已提交证据，不启动实验。")
elif command is None:
    print("本节需要额外环境准备；请使用上方链接中的 Terminal 流程。")
else:
    result = run_command(
        command,
        cwd=ctx.repo_root,
        output_dir=ctx.output_dir,
        label=f"{ctx.mode}-run",
        timeout_seconds=1200,
    )
    print({"passed": result.passed, "seconds": round(result.elapsed_seconds, 2), "log": result.log_path.name})


## 与预测对照、一般规律与边界

请回答：你的预测哪一部分被支持，哪一部分被推翻？当前证据只适用于哪些硬件、shape、消息大小或软件版本？

完成实验后再阅读[正式报告](../03_distributed_training/experiments/05_nccl_scaling.md)。正式报告是结论来源，Notebook 只是交互式观察层。

### 检查题

1. correctness 是否先于性能成立？
2. 当前指标的单位、重复方式和来源是什么？
3. `unavailable`、`failed` 与数值 0 为什么不能混为一谈？

下一步：[打开下一章](04_nccl_latency_bandwidth.ipynb)。
